# Declared and operative builders

This notebook shows what an addition does to a stage, on the consumption and portfolio stages of the portfolio–consumption application. Each stage file writes the maximization alone, with no marginal value and no candidate equation. The methods file beside each stage file, `methods.bl`, is an addition written in the language of the stage file: its blocks name the stage's blocks and add to them the lines that compute the policy and state the derivatives of the values, and its typing entries name the method, its grid and its handler at places of the stage. Binding it with `methodize` changes the operative builders, `stage.builder`, which hold what the addition adds; the declared builders, `stage.declared`, the operators the stage file's blocks denote, stay as they were. The stages are then joined in the two orders of a period, and the notebook shows the requirement that the endogenous grid method places on the stage that supplies the consumption stage's continuation. The notebook constructs and inspects; it solves nothing.

The files it reads are those of `applications/cons-port/declaration/`: `stages/cons/cons.bl`, `stages/port/port.bl`, `stages/cons/methods.bl`, `stages/port/methods.bl`, `stages/port/methods.without_marginals.bl` and the two recipes `spec.cons-port.yml` and `spec.port-cons.yml`, with the two recipes under `notebooks/without-portfolio-marginals/`.

The first cell imports the set-up module `checkout.py` beside this notebook, the module of the tutorials collection: it finds the checkout, puts its package first on the path and sets the working directory to the repository root, so that every path below is the path the repository gives.

In [1]:
import checkout
from bellman.stage import elaborate_stage, load_stage
from bellman.trellis import load_recipe, elaborate_trellis, methodize_trellis, calibrate_trellis, configure_trellis, derivative_missing

## 1 The consumption stage file

The household arrives with resources $\mathrm m$, consumes $\mathrm c$ out of them and saves $\mathrm a = \mathrm m_d - \mathrm c$. The policy block writes the argmax of $u(c) + V_\succ(a)$ over the feasible set $\Gamma(m_d) = (0, m_d]$ and nothing else.

In [2]:
print(open("applications/cons-port/declaration/stages/cons/cons.bl").read())

@stage: cons
Xm := ℝ₊₊
Xa := ℝ₊
ρ : (1, inf)
𝑢(c : ℝ₊₊) := c^(1-ρ)/(1-ρ)
Γ(x : ℝ₊₊) := (0, x]

[id=dcsn_to_cntn, (m : Xm, c : ℝ₊₊ @ !ctrl) : !dcsn -> (a : Xa) : !cntn] {
    a = m - c
}

[id=cntn_to_dcsn, (V[>] : ℝ @ !vlu, c) : !cntn -> (V : ℝ @ !vlu) : !dcsn] {
    V = 𝑢(c) + V[>]
}

[id=policy, (V[>]) : !cntn -> (c) : !dcsn] {
    c = argmax_{cp : Γ(m)}(𝑢(cp) + V[>]) :: [id=optimality]
}



## 2 The two stages

`elaborate_stage` reads a stage file into a stage, with nothing bound.

In [3]:
cons = elaborate_stage(load_stage("applications/cons-port/declaration/stages/cons/cons.bl"))

In [4]:
cons

ElaboratedStage('cons', symbolic)

The portfolio stage chooses the risky share $\varsigma$ of the savings before the return $\Psi$ and income $\theta$ are drawn; its policy block also writes the argmax alone.

In [5]:
port = elaborate_stage(load_stage("applications/cons-port/declaration/stages/port/port.bl"))

In [6]:
port

ElaboratedStage('port', symbolic)

## 3 The declared policy builder

`cons.declared` answers the declared builders. The policy builder's card lists its four operations: the Pullback of the continuation value $V_\succ$ along the law of motion, the Aggregation that forms the objective, the Argmax over the feasible set, and the Evaluation, the identity into the declared set of policies. With nothing bound, `cons.builder.policy` is the same builder.

In [7]:
cons.declared.policy.denote()

STAGE cons / POLICY BUILDER policy
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Signature
  in   𝑉≻  : Xa → ℝ
       ──────────────────────────────────────────────────────────────────
  out  𝑐   : Xm → ℝ₊₊,  𝑐(m) ∈ Γ(m)

Operations
  1  Pullback
     in: 𝑉≻                             out: 𝑉≻ ∘ 𝑔≻
     ────────────────────────────────────────────────────────────────────
     rule: (𝑉≻ ∘ 𝑔≻)(m, c) = 𝑉≻(𝑔≻(m, c))

  2  optimality:Aggregation
     in: (1)                            out: (m, c) ↦ 𝑢(c) + 𝑉≻(𝑔≻(m, c))
     ────────────────────────────────────────────────────────────────────
     rule: (m, c) ↦ 𝑢(c) + 𝑉≻(𝑔≻(m, c))

  3  optimality:Argmax
     in: (2)                            out: 𝑐
     ────────────────────────────────────────────────────────────────────
     rule: 𝑐(m) ∈ argmax_{cp ∈ Γ(m)} [𝑢(cp) + 𝑉≻(𝑔≻(m, cp))]

  4  Evaluation
     in: 𝑐                              out: 𝑐
     ───────────────────────────────────────────────────────────

The decision builder maps the continuation value to the decision value. It states no derivative of its value, since the stage file writes no derivative line.

In [8]:
cons.declared["cntn", "dcsn"].denote()

STAGE cons / BACKWARD BUILDER cntn → dcsn (cntn_to_dcsn)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Signature
  in   𝑉≻  : Xa → ℝ
       ───────────────────────────────────────────────────
  out  𝑉   : Xm → ℝ

At
  𝑐  : Xm → ℝ₊₊,  𝑐(m) ∈ Γ(m)

Operations
  1  Pullback
     in: 𝑉≻                             out: 𝑉≻ ∘ 𝑔≻(·, 𝑐)
     ─────────────────────────────────────────────────────
     rule: (𝑉≻ ∘ 𝑔≻(·, 𝑐))(m) = 𝑉≻(𝑔≻(m, 𝑐(m)))

  2  Aggregation
     in: (1)                            out: 𝑉
     ─────────────────────────────────────────────────────
     rule: 𝑉(m) = 𝑢(𝑐(m)) + 𝑉≻(𝑔≻(m, 𝑐(m)))

  3  Evaluation
     in: 𝑉                              out: 𝑉
     ─────────────────────────────────────────────────────
     rule: 𝑉 : Xm → ℝ

Conditions
  Aggregation  𝑉(m) ∈ ℝ
──────────────────────────────────────────────────────────────────────────────
Source  cons.bl
  13  V = 𝑢(c) + V[>]

Definitions referenced
  Xm    cons:set:Xm
  Xa    cons:set:Xa

## 4 The addition

The addition `methods.bl` beside the stage file opens with `@methods: cons`, writes the marginal utility of the stage's $u$ and its inverse in closed form, $u'(c) = c^{-\rho}$ and $(u')^{-1}(x) = x^{-1/\rho}$, declares the six settings its grids read, and holds two blocks that name the stage's blocks by their ids. The block `policy` types the policy builder with the endogenous grid method, `method : !egm`, with the application's handler `cons-port:egm` and with its grid of savings, and writes the method's three lines: the candidate consumption at each saving with its Kuhn–Tucker clause, the resources at which that saving is chosen, and the policy read on those resources. The block `cntn_to_dcsn` types the Evaluation of the decision value builder with the representation `!Linear` on the grid of resources, `evaluate.method` and `evaluate.grid`, and appends the envelope condition $D_{m}V = u'(c)$, the derivative of the decision value in the resources; the value and its stated derivative share that Evaluation and its one representation.

In [9]:
print(open("applications/cons-port/declaration/stages/cons/methods.bl").read())

@methods: cons
# The methods of the consumption stage at the occurrences of the two recipes,
# spec.cons-port.yml and spec.port-cons.yml, whose settings file
# settings/cons_wide.yml values multi-exponential grids: the endogenous grid
# method on the policy builder, realized by the function egm of
# code/cons_port_tool.py on the savings grid, with its three lines, the inverse
# Euler equation, the endogenous grid of resources and the identification of
# the policy; the envelope line of the decision value; and one linear
# representation of that value and of its stated derivative on the grid of
# resources. The marginal utility and its inverse are defined in closed form,
# so that nothing is differentiated. The three lines of the policy block stand
# in place of the maximization at the operative level, and stage.declared keeps
# the maximization. Each grid is its own typing entry at its place, the word
# !expmult its spacing; methods.reference.bl writes the same methods on evenly
# spac

## 5 Methodization with the addition

`methodize` reads the addition with the stage file: the stage is elaborated anew from the union of the two, bound in place, and returned.

In [10]:
cons.methodize("applications/cons-port/declaration/stages/cons/methods.bl")

ElaboratedStage('cons', methods)

`methods` lists the places the addition types, each with its method word, its grid as the entry writes it and its handler: `!egm` on the policy builder, and the representation `!Linear` at the Evaluation of the decision value builder, where the value and its stated derivative are stored.

In [11]:
cons.methods

cntn_to_dcsn.evaluate   !Linear   !Cartesian(m=(m_min, m_max, n_m, !expmult))
policy                  !egm   !Cartesian(a=(a_min, a_max, n_a, !expmult))   "cons-port:egm"

## 6 The policy builder, declared and operative

The declared policy builder is the one printed in section 3, line for line.

In [12]:
cons.declared.policy.denote()

STAGE cons / POLICY BUILDER policy
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Signature
  in   𝑉≻  : Xa → ℝ
       ──────────────────────────────────────────────────────────────────
  out  𝑐   : Xm → ℝ₊₊,  𝑐(m) ∈ Γ(m)

Operations
  1  Pullback
     in: 𝑉≻                             out: 𝑉≻ ∘ 𝑔≻
     ────────────────────────────────────────────────────────────────────
     rule: (𝑉≻ ∘ 𝑔≻)(m, c) = 𝑉≻(𝑔≻(m, c))

  2  optimality:Aggregation
     in: (1)                            out: (m, c) ↦ 𝑢(c) + 𝑉≻(𝑔≻(m, c))
     ────────────────────────────────────────────────────────────────────
     rule: (m, c) ↦ 𝑢(c) + 𝑉≻(𝑔≻(m, c))

  3  optimality:Argmax
     in: (2)                            out: 𝑐
     ────────────────────────────────────────────────────────────────────
     rule: 𝑐(m) ∈ argmax_{cp ∈ Γ(m)} [𝑢(cp) + 𝑉≻(𝑔≻(m, cp))]

  4  Evaluation
     in: 𝑐                              out: 𝑐
     ───────────────────────────────────────────────────────────

The operative policy builder reads the derivative of the continuation value, $D_{a}V_\succ$, by the derivative term of its candidate line. Its card prints the stage file's argmax under `Declared`, the statement the construction is to reach, then the addition's rows, the conditions by name, the method with its grid and its handler, and the lines of both files under `Source`.

In [13]:
cons.builder.policy.denote()

STAGE cons / POLICY BUILDER policy
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Signature
  in   𝑉≻  : Xa → ℝ
       ────────────────────────────────────────
  out  𝑐   : Xm → ℝ₊₊,  𝑐(m) ∈ Γ(m)

Declared
  𝑐(m) ∈ argmax_{cp ∈ Γ(m)} [𝑢(cp) + 𝑉≻(𝑔≻(m, cp))]

Operations
  1  InvEuler:Aggregation
     in: 𝑉≻                             out: 𝑐≻
     ──────────────────────────────────────────
     rule: 𝑐≻(a) ≤ 𝑑𝑢𝑖𝑛𝑣(D_{a}𝑉≻(a)) ⟂ a ≥ 0

  2  grid:Aggregation
     in: 𝑐≻                             out: 𝑚≻
     ──────────────────────────────────────────
     rule: 𝑚≻(a) = a + 𝑐≻(a)

  3  identification:Pullback
     in: 𝑐≻, 𝑚≻                         out: 𝑐
     ──────────────────────────────────────────
     rule: 𝑐(𝑚≻(a)) = 𝑐≻(a)                                          on 𝑚≻(Xa)

  4  Evaluation
     in: 𝑐                              out: 𝑐
     ──────────────────────────────────────────
     rule: 𝑐 : Xm → ℝ₊₊,  𝑐(m) ∈ Γ(m)

Conditions
  grid:Aggregatio

## 7 The decision builder, declared and operative

The declared decision builder is unchanged by the binding.

In [14]:
cons.declared["cntn", "dcsn"].denote()

STAGE cons / BACKWARD BUILDER cntn → dcsn (cntn_to_dcsn)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Signature
  in   𝑉≻  : Xa → ℝ
       ───────────────────────────────────────────────────
  out  𝑉   : Xm → ℝ

At
  𝑐  : Xm → ℝ₊₊,  𝑐(m) ∈ Γ(m)

Operations
  1  Pullback
     in: 𝑉≻                             out: 𝑉≻ ∘ 𝑔≻(·, 𝑐)
     ─────────────────────────────────────────────────────
     rule: (𝑉≻ ∘ 𝑔≻(·, 𝑐))(m) = 𝑉≻(𝑔≻(m, 𝑐(m)))

  2  Aggregation
     in: (1)                            out: 𝑉
     ─────────────────────────────────────────────────────
     rule: 𝑉(m) = 𝑢(𝑐(m)) + 𝑉≻(𝑔≻(m, 𝑐(m)))

  3  Evaluation
     in: 𝑉                              out: 𝑉
     ─────────────────────────────────────────────────────
     rule: 𝑉 : Xm → ℝ

Conditions
  Aggregation  𝑉(m) ∈ ℝ
──────────────────────────────────────────────────────────────────────────────
Source  cons.bl
  13  V = 𝑢(c) + V[>]

Definitions referenced
  Xm    cons:set:Xm
  Xa    cons:set:Xa

The operative decision builder states the derivative of the value it writes: the envelope line stands at its Evaluation beside the representation, the signature is unchanged, and the `Source` section quotes the line from the addition.

In [15]:
cons.builder["cntn", "dcsn"].denote()

STAGE cons / BACKWARD BUILDER cntn → dcsn (cntn_to_dcsn)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Signature
  in   𝑉≻  : Xa → ℝ
       ───────────────────────────────────────────────────
  out  𝑉   : Xm → ℝ

At
  𝑐  : Xm → ℝ₊₊,  𝑐(m) ∈ Γ(m)

Operations
  1  Pullback
     in: 𝑉≻                             out: 𝑉≻ ∘ 𝑔≻(·, 𝑐)
     ─────────────────────────────────────────────────────
     rule: (𝑉≻ ∘ 𝑔≻(·, 𝑐))(m) = 𝑉≻(𝑔≻(m, 𝑐(m)))

  2  Aggregation
     in: (1)                            out: 𝑉
     ─────────────────────────────────────────────────────
     rule: 𝑉(m) = 𝑢(𝑐(m)) + 𝑉≻(𝑔≻(m, 𝑐(m)))

  3  Evaluation
     in: 𝑉                              out: 𝑉
     ─────────────────────────────────────────────────────
     rule: 𝑉 : Xm → ℝ
           D_{m}𝑉(m) = 𝑑𝑢(𝑐(m))

Conditions
  Aggregation  𝑉(m) ∈ ℝ

Method
  !Linear   !Cartesian(m=(m_min, m_max, n_m, !expmult))  cntn_to_dcsn.evaluate
─────────────────────────────────────────────────────────────

## 8 The conditions

`obligations()` lists the condition of every rule applied, by its name. Among them are `egm-eligibility`, under which the policy the addition's block computes satisfies the declared argmax, `function-argument-domain-inclusion`, that the derivative the candidate line passes to the inverse of the marginal utility lies in its domain, and `marginal-reading`, the reading of the author's derivative line. A condition marked required remains unresolved; its presence does not certify the numerical method.

In [16]:
cons.obligations()

aggregation-admissibility            V(m) ∈ ℝ
marginal-reading                     the right side of each derivative line equals the derivative, in the named coordinate, of the value the builder writes, under the envelope conditions: concavity of the continuation value, interiority of the maximizer or the multiplier's term at a binding constraint, differentiability of the reward in the control, a measurable selection of the policy, and differentiability of the value at the point read; listed and not checked
aggregation-admissibility            m≻(a) ∈ Xm
range-compatibility                  m − c(m) ∈ Xa   [established]
function-argument-domain-inclusion   The argument for x of cons:function:𝑑𝑢𝑖𝑛𝑣 at policy.InvEuler, from ℝ, must belong to ℝ₊₊.
egm-eligibility                      the construction satisfies the declared line where the continuation value 𝑉≻ is concave and continuously differentiable above the savings bound, with a positive right derivative at the bound, finite or infini

## 9 The portfolio stage's addition

The block `policy` types the policy builder with `!time_iteration`, the handler `share_search` and its grid of shares, and computes the share from its first-order condition, the expected excess return weighted by the derivative of the continuation value; the block `cntn_to_dcsn` types the Evaluation with `!Linear` on the grid of capital and appends the derivative of the decision value in capital. Gauss-Hermite quadrature is typed at the expectation of each block, with one node count per shock, and the forward law is drawn by Monte Carlo.

In [17]:
print(open("applications/cons-port/declaration/stages/port/methods.bl").read())

@methods: port
# The methods of the portfolio stage at the occurrences of the two recipes and
# of trellis.port-cons-vfi.yml, whose settings file settings/port_wide.yml
# values a multi-exponential capital grid: time iteration on the policy
# builder with the first-order condition of the share as its line, realized by
# the function share_search of code/cons_port_tool.py, which compares n_ς
# shares on the feasible set at each point of the capital grid; Gauss-Hermite
# quadrature at every occurrence of the expectation; the envelope line of the
# decision value; the linear representation of the decision value and of its
# stated derivative on the capital grid; and the forward law drawn by Monte
# Carlo. The line euler stands in place of the maximization at the operative
# level, and stage.declared keeps the maximization. Each grid is its own
# typing entry at its place; methods.reference.bl writes the fixed reference's
# handlers on an evenly spaced grid.

n_k : ℕ              :: ["numb

In [18]:
port.methodize("applications/cons-port/declaration/stages/port/methods.bl")

ElaboratedStage('port', methods)

The declared policy builder of the portfolio stage holds the argmax over the share; the operative one holds the first-order condition, under the method `!time_iteration`, with the argmax printed under `Declared`. The handler `cons-port:share_search` beside the method chooses the share by the search over the declared objective, as the application's tool realizes it.

In [19]:
port.declared.policy.denote()

STAGE port / POLICY BUILDER policy
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Signature
  in   𝑉≻  : ℝ₊₊ → ℝ
       ─────────────────────────────────────────────
  out  𝜍   : ℝ₊ → [0, 1],  𝜍(k) ∈ Γ_ς(k)

Operations
  1  Pullback
     in: 𝑉≻                             out: 𝑉≻ ∘ 𝑔≻
     ───────────────────────────────────────────────
     rule: (𝑉≻ ∘ 𝑔≻)(k, ς, Ψ, θ) = 𝑉≻(𝑔≻(k, ς, Ψ, θ))

  2  objective:Expectation:1
     in: (1)
     out: (k, ς) ↦ E_{Ψ, θ}[𝑉≻(𝑔≻(k, ς, Ψ, θ))]
     ───────────────────────────────────────────────
     rule: (2)(k, ς) = E_{Ψ, θ}[𝑉≻(𝑔≻(k, ς, Ψ, θ))]

  3  objective:Argmax
     in: (2)                            out: 𝜍
     ───────────────────────────────────────────────
     rule: 𝜍(k) ∈ argmax_{ς ∈ Γ_ς(k)} [E_{Ψ, θ}[𝑉≻(𝑔≻(k, ς, Ψ, θ))]]

  4  Evaluation
     in: 𝜍                              out: 𝜍
     ───────────────────────────────────────────────
     rule: 𝜍 : ℝ₊ → [0, 1],  𝜍(k) ∈ Γ_ς(k)
───────────────────────────

In [20]:
port.builder.policy.denote()

STAGE port / POLICY BUILDER policy
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Signature
  in   𝑉≻  : ℝ₊₊ → ℝ
       ───────────────────────────────────────────────────
  out  𝜍   : ℝ₊ → [0, 1],  𝜍(k) ∈ Γ_ς(k)

Declared
  𝜍(k) ∈ argmax_{ς ∈ Γ_ς(k)} [E_{Ψ, θ}[𝑉≻(𝑔≻(k, ς, Ψ, θ))]]

Operations
  1  Pullback
     in: 𝑉≻                             out: 𝑉≻ ∘ 𝑔≻
     ─────────────────────────────────────────────────────
     rule: (𝑉≻ ∘ 𝑔≻)(k, ς, Ψ, θ) = 𝑉≻(𝑔≻(k, ς, Ψ, θ))

  2  euler:Aggregation:1
     in: (1)
     out: (k, ς, Ψ, θ) ↦ D_{w}𝑉≻(𝑔≻(k, ς, Ψ, θ)) · (Ψ − R)
     ─────────────────────────────────────────────────────
     rule: (k, ς, Ψ, θ) ↦ D_{w}𝑉≻(𝑔≻(k, ς, Ψ, θ)) · (Ψ − R)

  3  euler:Expectation:1
     in: (2)                            out: r₁
     ─────────────────────────────────────────────────────
     rule: r₁ = E_{Ψ, θ}[D_{w}𝑉≻(𝑔≻(k, ς, Ψ, θ)) · (Ψ − R)]

  4  euler:Aggregation
     in: r₁                             out: 𝜍
     ───────

## 10 The two orders

Each recipe joins the consumption, discount and portfolio stages into a period and repeats it; `load_recipe` reads the recipe. The four construction steps follow separately: `elaborate`, `methodize`, `calibrate`, and `configure`. In the first order consumption is chosen first and the share on the savings.

In [21]:
cons_first_syntax = load_recipe("applications/cons-port/declaration", "spec.cons-port.yml")
cons_first_symbolic = elaborate_trellis(cons_first_syntax)
cons_first_methodized = methodize_trellis(cons_first_symbolic, cons_first_syntax)
cons_first_calibrated = calibrate_trellis(cons_first_methodized, cons_first_syntax)
cons_first = configure_trellis(cons_first_calibrated, cons_first_syntax)


The consumption stage's block reads $D_{a}V_\succ$, so the value that the stage supplying its continuation exports must carry its derivative in the savings, and so on along the period. `derivative_missing` lists each derivative reading that no route meets, naming the stage, the value and the coordinate. Every value here carries its derivative, by a stated line or by its representation, and the list is empty.

In [22]:
derivative_missing(cons_first)

()

In the second order the share is chosen first, on the arrival resources, and consumption on the resources the draws give.

In [23]:
port_first_syntax = load_recipe("applications/cons-port/declaration", "spec.port-cons.yml")
port_first_symbolic = elaborate_trellis(port_first_syntax)
port_first_methodized = methodize_trellis(port_first_symbolic, port_first_syntax)
port_first_calibrated = calibrate_trellis(port_first_methodized, port_first_syntax)
port_first = configure_trellis(port_first_calibrated, port_first_syntax)


In [24]:
derivative_missing(port_first)

()

## 11 Absent portfolio derivatives

The replacing selection `methods.without_marginals.bl` is the portfolio stage's addition without its envelope line and without its `!Linear` representation, which would evaluate the first derivative of the represented value, so that the value the stage exports carries no derivative in capital.

In [25]:
print(open("applications/cons-port/declaration/stages/port/methods.without_marginals.bl").read())

@methods: port
# The methods of the portfolio stage at the occurrences of the two recipes
# under notebooks/without-portfolio-marginals/: the methods of methods.bl
# without the envelope line and without the representation of the decision
# value, so that the value the stage exports carries no stated derivative. The
# block cntn_to_dcsn receives no line, so its one selection is a file-level
# line naming the expectation by its line.

n_ς : ℕ              :: ["number of shares the search compares"]
n_Ψ_nodes : ℕ        :: ["quadrature nodes of the return shock"]
n_θ_nodes : ℕ        :: ["quadrature nodes of the income shock"]
n_sim : ℕ            :: ["number of simulated paths"]
sim_seed : ℕ         :: ["seed of the simulation"]

[id=policy, policy.method : !time_iteration
    | policy.handler : "cons-port:share_search"
    | policy.grid : !Cartesian(ς=(Γ_ς, n_ς))
    | E_Ψ_θ.method : !gauss_hermite(n_Ψ=n_Ψ_nodes, n_θ=n_θ_nodes)
    | E_Ψ_θ.handler : "cons-port:gauss_hermite"] {
    ς =

The two recipes under `notebooks/without-portfolio-marginals/` are the folder's recipes with this selection in place of `methods.bl`. Consumption first:

In [26]:
cons_first_without_syntax = load_recipe("applications/cons-port/declaration", "../notebooks/without-portfolio-marginals/spec.cons-port.yml")
cons_first_without_symbolic = elaborate_trellis(cons_first_without_syntax)
cons_first_without_methodized = methodize_trellis(cons_first_without_symbolic, cons_first_without_syntax)
cons_first_without_calibrated = calibrate_trellis(cons_first_without_methodized, cons_first_without_syntax)
cons_first_without = configure_trellis(cons_first_without_calibrated, cons_first_without_syntax)


Each entry names the portfolio occurrence, its value and the coordinate `k`, the reading `D_{a}V[>]` of the consumption stage's block that reaches it through the discount stage's pass-through, and says what to write. The stages bind and nothing is refused: a missing derivative is a requirement reported on the trellis, not a refusal of the method.

In [27]:
derivative_missing(cons_first_without)

(Diagnostic(rule='derivative-missing', address="periods[0].stages['port']", message="stage 'port' at periods[0].port supplies, through the pass-through of disc, the continuation of periods[0].cons, whose block 'policy' reads `D_{a}V[>]`, and its value `V` carries no derivative in k; state it with a derivative line of the builder that writes the value of stage 'port', in its stage file or in an addition, or type at that builder's Evaluation, `cntn_to_dcsn.evaluate.method : …`, a representation whose row of the catalogue evaluates it", candidates=('port:function:dcsn:V', 'k'), source_entries=(), depends_on=None, write=None),
 Diagnostic(rule='derivative-missing', address="periods[1].stages['port']", message="stage 'port' at periods[1].port supplies, through the pass-through of disc, the continuation of periods[1].cons, whose block 'policy' reads `D_{a}V[>]`, and its value `V` carries no derivative in k; state it with a derivative line of the builder that writes the value of stage 'port',

Portfolio first:

In [28]:
port_first_without_syntax = load_recipe("applications/cons-port/declaration", "../notebooks/without-portfolio-marginals/spec.port-cons.yml")
port_first_without_symbolic = elaborate_trellis(port_first_without_syntax)
port_first_without_methodized = methodize_trellis(port_first_without_symbolic, port_first_without_syntax)
port_first_without_calibrated = calibrate_trellis(port_first_without_methodized, port_first_without_syntax)
port_first_without = configure_trellis(port_first_without_calibrated, port_first_without_syntax)


In [29]:
derivative_missing(port_first_without)

(Diagnostic(rule='derivative-missing', address="periods[0].stages['port']", message="stage 'port' at periods[0].port supplies, through the pass-through of disc, the continuation of periods[1].cons, whose block 'policy' reads `D_{a}V[>]`, and its value `V` carries no derivative in k; state it with a derivative line of the builder that writes the value of stage 'port', in its stage file or in an addition, or type at that builder's Evaluation, `cntn_to_dcsn.evaluate.method : …`, a representation whose row of the catalogue evaluates it", candidates=('port:function:dcsn:V', 'k'), source_entries=(), depends_on=None, write=None),
 Diagnostic(rule='derivative-missing', address="periods[1].stages['port']", message="stage 'port' at periods[1].port supplies, through the pass-through of disc, the continuation of periods[2].cons, whose block 'policy' reads `D_{a}V[>]`, and its value `V` carries no derivative in k; state it with a derivative line of the builder that writes the value of stage 'port',

## 12 The two states in the Workbench

Bellman Workbench shows a stage file beside the graph it denotes. The layers button at the foot of the graph switches the builders shown. Off, the graph and the cards are the declared builders: the policy builder of the consumption stage holds the argmax, and the details name the bound method.

![The consumption stage in Bellman Workbench with the layers button off: the declared builders](images/layers-button-off.png)

On, they are the operative builders: the decision builder's Evaluation states the derivative $D_{m}V$, the envelope line the addition appends, beside the representation, which the picture labels `!Cartesian` and the addition now writes as `!Linear` with its grid `!Cartesian(m=(m_min, m_max, n_m, !expmult))`, and the details of the policy builder print its signature, its declared line and its conditions. The two pictures were taken on 22 October 2026, before the derivative term replaced the marginal field, and show the earlier labels.

![The consumption stage in Bellman Workbench with the layers button on: the operative builders](images/layers-button-on.png)

### Observation

### Decision